# Reinforcement Learning Based Trading Agent 

Follow the instructions step by step and fill in the TODOs


## 1. Install and Import Libraries

In [ ]:

# Uncomment only if needed
# !pip install yfinance numpy pandas matplotlib

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt


## 2. Download Market Data 

In [ ]:
# choose a stock symbol
symbol = "AAPL"

# download historical stock price data
data = yf.download(symbol, start="2019-01-01", end="2024-01-01")

# extract ONLY closing prices and flatten
prices = data["Close"].values.flatten()

print("Trading days:", len(prices))
print("Sample price:", prices[0], type(prices[0]))


C:\Users\Mohit\AppData\Local\Temp\ipykernel_30156\2571435805.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start="2019-01-01", end="2024-01-01")
[*********************100%***********************]  1 of 1 completed

Trading days: 1258
Sample price: 37.538814544677734 <class 'numpy.float64'>


## 3. Trading Environment

In [ ]:
class TradingEnv:
    def __init__(self, prices):
        self.prices = prices
        self.reset()

    def reset(self):
        self.t = 0
        self.cash = 10000
        self.stock = 0
        self.done = False
        return self._get_state()

    def _get_state(self):
        return np.array([self.prices[self.t], self.stock])

    def step(self, action):
        price = self.prices[self.t]

        # Buy
        if action == 1 and self.cash >= price and self.stock == 0:
            self.cash -= price
            self.stock = 1

        # Sell
        elif action == 2 and self.stock == 1:
            self.cash += price
            self.stock = 0

        # move to next step
        self.t += 1

        # termination
        if self.t >= len(self.prices) - 1:
            self.done = True

        reward = self.cash + self.stock * self.prices[self.t]
        return self._get_state(), reward, self.done


## 4. Q-Learning Setup

In [ ]:
Q = np.zeros((len(prices), 3))

alpha = 0.1
gamma = 0.95
epsilon = 0.1


## 5. Train the Agent

In [ ]:
env = TradingEnv(prices)
episodes = 50

for episode in range(episodes):
    env.reset()

    while not env.done:
        t = env.t

        if np.random.rand() < epsilon:
            action = np.random.randint(3)
        else:
            action = np.argmax(Q[t])

        _, reward, _ = env.step(action)

        Q[t, action] += alpha * (
            reward +
            gamma * np.max(Q[min(t + 1, len(prices) - 1)]) -
            Q[t, action]
        )

print("Training completed")


Training completed


## 6. Evaluate Trained Agent

In [ ]:
env = TradingEnv(prices)

while not env.done:
    t = env.t
    action = np.argmax(Q[t])
    env.step(action)

final_value = env.cash + env.stock * prices[-1]
print("Final portfolio value (RL):", final_value)


Final portfolio value (RL): 10178.851306915283


## 7. Buy and Hold Baseline

In [ ]:
buy_and_hold_value = 10000 - prices[0] + prices[-1]
print("Final portfolio value (Buy & Hold):", buy_and_hold_value)


Final portfolio value (Buy & Hold): 10153.189945220947
